# Stochastic Calculus from Zero · Episode 23: Feynman-Kac: expectations are PDEs

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> **The short rate** (illustrative): starts at 3.85%, pulled towards 3.50% with mean reversion 0.30 and volatility 1% a year (Vasicek). Made up for teaching.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. The theorem · 4. Why it's true · 5. The Vasicek bond · 6. Four answers · 7. Every starting rate at once · 8. This episode's question · 9. Exercises · 10. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 23,
    "quantlib_version": ql.__version__,
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, math, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)


# ---- Part 3 additions -------------------------------------------------------------------------------------------
#   Short rate (illustrative): r0 3.85%, long-run level 3.50%, mean reversion 0.30, volatility 1% (Vasicek).
def norm_cdf(x):
    """Standard normal distribution function, for numbers or numpy arrays."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x) / math.sqrt(2)))


def bs_call(s, k, t, r, vol):
    """Black-Scholes call price and delta (Series 3, Episode 13)."""
    d1 = (np.log(s / k) + (r + 0.5 * vol ** 2) * t) / (vol * np.sqrt(t))
    return s * norm_cdf(d1) - k * np.exp(-r * t) * norm_cdf(d1 - vol * np.sqrt(t)), norm_cdf(d1)


def ou_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of an Ornstein-Uhlenbeck (Vasicek) rate t years ahead."""
    return theta + (r0 - theta) * math.exp(-a * t), sigma * math.sqrt((1 - math.exp(-2 * a * t)) / (2 * a))


def ou_step(r, dt, a, theta, sigma, z):
    """One exact OU step of length dt from r, driven by standard normals z: no discretisation error."""
    return theta + (r - theta) * math.exp(-a * dt) + sigma * math.sqrt((1 - math.exp(-2 * a * dt)) / (2 * a)) * z


def vasicek_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price P = A exp(-B r) in the Vasicek model, tau years to maturity."""
    B = (1 - math.exp(-a * tau)) / a
    lnA = (theta - sigma ** 2 / (2 * a * a)) * (B - tau) - sigma ** 2 * B * B / (4 * a)
    return np.exp(lnA - B * np.asarray(r))


def cir_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of a CIR rate t years ahead."""
    e = math.exp(-a * t)
    var = r0 * sigma ** 2 / a * (e - e * e) + theta * sigma ** 2 / (2 * a) * (1 - e) ** 2
    return theta + (r0 - theta) * e, math.sqrt(var)


def cir_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price in the CIR model (Cox, Ingersoll and Ross, 1985)."""
    g = math.sqrt(a * a + 2 * sigma * sigma)
    den = (g + a) * (math.exp(g * tau) - 1) + 2 * g
    B = 2 * (math.exp(g * tau) - 1) / den
    A_ = (2 * g * math.exp((a + g) * tau / 2) / den) ** (2 * a * theta / sigma ** 2)
    return A_ * np.exp(-B * np.asarray(r))

## 2. Last episode's answer

The hedging error shrinks like $1/\sqrt n$: four times as often halves it (weekly 1.01, four times weekly 0.51).

In [ ]:
out["prev"] = {"answer": "halves"}

## 3. The theorem

Let $dX = \mu(t, X)\,dt + \sigma(t, X)\,dW$, a discount rate $r(t, x)$ and a payoff $g$. Then

$$u(t, x) = E\Big[e^{-\int_t^T r(s, X_s)\,ds}\,g(X_T) \;\Big|\; X_t = x\Big]$$

solves the **Feynman-Kac** PDE

$$u_t + \mu\,u_x + \tfrac12\sigma^2\,u_{xx} - r\,u = 0, \qquad u(T, x) = g(x).$$

(Kac, 1949, after Feynman's path integrals of 1948.) Series 3, Episode 24 used it for one formula, Black-Scholes. Here it is in general, and on an interest-rate product.

## 4. Why it's true

Let $D_t = e^{-\int_0^t r\,ds}$. By the tower property (Part 1), $M_t = D_t\,u(t, X_t) = E[D_T\,g(X_T) \mid \mathcal F_t]$ is a martingale. Itô-Doeblin (Episode 15) with the product rule:

$$dM = D_t\big(u_t + \mu u_x + \tfrac12\sigma^2 u_{xx} - r u\big)\,dt + D_t\,\sigma u_x\,dW .$$

A martingale has no $dt$ term (Part 2), so the bracket is zero: that is the PDE.

## 5. The Vasicek bond

A zero-coupon bond paying 1 at $T$: $X = r$ (the Vasicek short rate, now under the pricing measure), the discount rate is $r$ itself, and $g = 1$:

$$P(t, r) = E\Big[e^{-\int_t^T r_s\,ds} \,\Big|\, r_t = r\Big], \qquad P_t + a(\theta - r)P_r + \tfrac12\sigma^2P_{rr} - rP = 0,\quad P(T, r) = 1 .$$

Guess $P = A(\tau)e^{-B(\tau)r}$ with $\tau = T - t$ (an **affine** model): the PDE splits into ordinary equations, $B' = 1 - aB$ and one for $\ln A$, with the solution in `vasicek_bond`.

In [ ]:
R0, THETA, A, SIG = 0.0385, 0.035, 0.30, 0.01
TB = 5.0
closed = float(vasicek_bond(TB, R0, A, THETA, SIG))
qlv = ql.Vasicek(R0, A, THETA, SIG).discountBond(0.0, TB, R0)
assert abs(closed - qlv) < 1e-10
out["inputs"] = {"maturity": TB, "years": {"kac": 1949, "feynman": 1948}}
print(f"closed form {closed:.6f}, QuantLib Vasicek {qlv:.6f}")

## 6. Four answers

**Monte Carlo**: 50,000 paths with exact weekly steps (`ou_step`), the integral of $r$ by the trapezoid rule, then average $e^{-\int r}$.

**PDE**: Crank-Nicolson on a grid of rates from $-10\%$ to $20\%$, stepping backwards from $P(T, r) = 1$. At the edges of the grid, far from today's rate, we drop the $P_r$ and $P_{rr}$ terms.

In [ ]:
rng = np.random.default_rng(seed)
n_paths, steps = 50_000, int(TB * 52)
dt = TB / steps
r = np.full(n_paths, R0); integral = np.zeros(n_paths)
for _ in range(steps):
    rn = ou_step(r, dt, A, THETA, SIG, rng.standard_normal(n_paths))
    integral += 0.5 * (r + rn) * dt
    r = rn
disc = np.exp(-integral)
mc, mc_se = float(disc.mean()), float(disc.std() / math.sqrt(n_paths))

def pde_bond(maturity, points=301, time_steps=500, lo=-0.10, hi=0.20):
    '''Crank-Nicolson for P_tau = a(theta - r) P_r + 1/2 sigma^2 P_rr - r P, P(tau = 0) = 1. Returns the grid and P.'''
    rg = np.linspace(lo, hi, points); h = rg[1] - rg[0]; k = maturity / time_steps
    drift, diff = A * (THETA - rg), 0.5 * SIG ** 2
    L = (np.diag(-2 * diff / h ** 2 - rg) + np.diag((diff / h ** 2 - drift / (2 * h))[1:], -1)
         + np.diag((diff / h ** 2 + drift / (2 * h))[:-1], 1))
    L[0, :] = 0; L[-1, :] = 0; L[0, 0] = -rg[0]; L[-1, -1] = -rg[-1]
    I = np.eye(points)
    step = np.linalg.solve(I - 0.5 * k * L, I + 0.5 * k * L)
    p = np.ones(points)
    for _ in range(time_steps):
        p = step @ p
    return rg, p

rg, pg = pde_bond(TB)
pde = float(np.interp(R0, rg, pg))
four = {"closed": closed, "ql": qlv, "mc": mc, "mc_se": mc_se, "pde": pde, "paths": n_paths}
assert abs(mc - closed) < 3 * mc_se + 2e-5 and abs(pde - closed) < 1e-4
out["four"] = four
pd.Series(four)

## 7. Every starting rate at once

The PDE solves for every starting rate on the grid in one go; Monte Carlo answers one starting point per run.

In [ ]:
mask = (rg >= -0.02) & (rg <= 0.10)
closed_curve = vasicek_bond(TB, rg[mask], A, THETA, SIG)
out["curve"] = {"r_pct": (rg[mask] * 100).round(3).tolist(), "pde": pg[mask].round(6).tolist(),
                "closed": np.asarray(closed_curve).round(6).tolist(), "max_diff": float(np.abs(pg[mask] - closed_curve).max()),
                "r0_pct": R0 * 100, "ticks": [{"x": x, "label": f"{x}%"} for x in range(-2, 11, 2)]}
assert out["curve"]["max_diff"] < 2e-4
print(f"largest PDE error between -2% and 10%: {out['curve']['max_diff']:.1e}")

## 8. This episode's question

With $\sigma = 0$ the rate follows $r(t) = \theta + (r_0 - \theta)e^{-at}$ exactly, and the bond is $\exp\big(-\theta T - (r_0 - \theta)(1 - e^{-aT})/a\big)$.

In [ ]:
det = math.exp(-THETA * TB - (R0 - THETA) * (1 - math.exp(-A * TB)) / A)
assert abs(det - float(vasicek_bond(TB, R0, A, THETA, 1e-9))) < 1e-9
out["question"] = {"sigma0": det, "with_vol": closed, "convexity_bp": (closed - det) * 1e4}
out["question"]

## 9. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Repeat the martingale argument without discounting: if $u(t, x) = E[g(X_T) \mid X_t = x]$, which PDE does $u$ solve?

<details><summary>Answer</summary>

$u(t, X_t)$ is itself a martingale; its $dt$ term must vanish: $u_t + \mu u_x + \tfrac12\sigma^2 u_{xx} = 0$, $u(T, x) = g(x)$ (the backward Kolmogorov equation).

</details>

**Exercise 2 (By hand).** Substitute $P = A(\tau)e^{-B(\tau)r}$ into the Vasicek PDE (with $P_t = -P_\tau$). Show that $B' = 1 - aB$, $B(0) = 0$, and solve it.

<details><summary>Answer</summary>

$P_r = -BP$, $P_{rr} = B^2P$, $P_t = -P_\tau = -(A'/A - B'r)P$. Divide the PDE by $P$: $-A'/A + B'r - a\theta B + aBr + \tfrac12\sigma^2B^2 - r = 0$. The terms in $r$ give $B' + aB - 1 = 0$, so $B' = 1 - aB$ and, with $B(0) = 0$, $B(\tau) = (1 - e^{-a\tau})/a$. The rest gives $(\ln A)' = -a\theta B + \tfrac12\sigma^2B^2$.

</details>

**Exercise 3 (By hand).** Why does adding volatility ($\sigma > 0$) make the bond worth more than the deterministic one?

<details><summary>Answer</summary>

$e^{-x}$ is convex, so by Jensen $E[e^{-\int r}] \ge e^{-E[\int r]}$, and $E[\int r]$ is the same with or without $\sigma$ in Vasicek. Here the gain is small (see §8).

</details>

**Exercise 4 (In code).** Refine the PDE grid (101, 201, 401 rate points) and watch the error to the closed form fall.

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
errs = []
for pts in (101, 201, 401):
    g_, p_ = pde_bond(TB, points=pts)
    errs.append(abs(float(np.interp(R0, g_, p_)) - closed))
assert errs[-1] < errs[0]
print([f"{e:.1e}" for e in errs])

**Exercise 5 (In code).** Price a 10-year bond three ways: closed form, QuantLib, PDE.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
c10 = float(vasicek_bond(10.0, R0, A, THETA, SIG))
q10 = ql.Vasicek(R0, A, THETA, SIG).discountBond(0.0, 10.0, R0)
g10, p10 = pde_bond(10.0, time_steps=1000)
d10 = float(np.interp(R0, g10, p10))
assert abs(c10 - q10) < 1e-10 and abs(d10 - c10) < 2e-4
print(f"10-year bond: closed {c10:.6f}, QuantLib {q10:.6f}, PDE {d10:.6f}")

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")